### Upper Bounds on $n$ for $k=9$ via Linear Forms in Logarithms

In Section 8.2 of the manuscript, the cases $(C,D) \in \{(5,3), (1,15)\}$ require upper bounds on the exponent $n$ to reduce the problem to a finite search space. This is achieved by deriving a linear form in two logarithms from the equation $2(Cc^n \pm 1)^2 \mp 3Cc^n = Dd^n$.

The script below applies Laurent's Theorem (Lemma 4.2) to calculate absolute upper bounds on $n$. It iterates over a grid of parameters for $\mu \in (1/3,1]$ and $\rho \in [2, 20]$ to optimize the resulting bounds. Given $c \geq 11$ and the assumption $n > 1000$, the script computes the required heights, evaluates the inequalities, and outputs the optimal bound $n_0$ for each evaluated $\mu$.

In [1]:
# =============================================================================
# Baker's Method: Upper Bounds on n for k = 9
# =============================================================================

var('n', 'c', 'logc')
RR = RealField(1000)

# --- Laurent's Theorem Parameters ---

def sigma(mu):
    return ((1 + 2 * mu - mu^2) / 2).n()

def lambda_val(rho, mu):
    return sigma(mu) * log(rho)

def h_val(rho, mu, a1, a2, deg, hid, n_lb):
    if hid == 0:
        return deg * (log(n / a2 + 1 / a1) + log(lambda_val(rho, mu)) + 1.75) + 0.06
    elif hid == 1:
        return max(lambda_val(rho, mu), (deg * log(2) / 2))

def H_val(rho, mu, a1, a2, deg, hid, n_lb):
    return h_val(rho, mu, a1, a2, deg, hid, n_lb) / lambda_val(rho, mu) + 1 / sigma(mu)

def omega(rho, mu, a1, a2, deg, hid, n_lb):
    return 2 * (1 + sqrt(1 + 1 / (4 * H_val(rho, mu, a1, a2, deg, hid, n_lb)^2)))

def theta(rho, mu, a1, a2, deg, hid, n_lb):
    H = H_val(rho, mu, a1, a2, deg, hid, n_lb)
    return omega(rho, mu, a1, a2, deg, hid, n_lb) / 2 - 1 + 1 / (2 * H)

def verify_lambda(rho, mu, a1, a2, c_lb):
    a_prod = (a1 * a2)(logc=c_lb).n()
    l_sq = (lambda_val(rho, mu)^2).n()
    return a_prod >= l_sq

def a_vals(rho, mu, A, Ceq, deg, n_lb):
    # Heights as derived in Section 8.2
    a1 = ((rho - 1) / n_lb) * log(2 * max(A + 1, 2 * Ceq^2 + 1) / 3) + 2 * logc
    a2 = (rho - 1) * abs(log(2 * A^2 / Ceq)) + 2 * log(max(2 * A^2, Ceq))
    return (a1, a2)

def C_val(rho, mu, A, Ceq, deg, hid, n_lb):
    a1, a2 = a_vals(rho, mu, A, Ceq, deg, n_lb)
    l_val = lambda_val(rho, mu)
    H = H_val(rho, mu, a1, a2, deg, hid, n_lb)
    w_val = omega(rho, mu, a1, a2, deg, hid, n_lb)
    
    pt1 = mu / (l_val^3 * sigma(mu))
    pt2 = w_val / 6
    pt3 = w_val^2 / 9 + (4 / 3) * (1 / a1 + 1 / a2) * l_val * w_val / H
    pt4_num = 8 * l_val * w_val^(5/4) * theta(rho, mu, a1, a2, deg, hid, n_lb)^(1/4)
    pt4_denom = 3 * sqrt(a1 * a2 * H)
    
    return pt1 * (pt2 + 1/2 * sqrt(pt3 + pt4_num / pt4_denom))^2

def C_prime(rho, mu, A, Ceq, deg, hid, n_lb):
    C = C_val(rho, mu, A, Ceq, deg, hid, n_lb)
    a1, a2 = a_vals(rho, mu, A, Ceq, deg, n_lb)
    l_val = lambda_val(rho, mu)
    
    num = C * sigma(mu) * omega(rho, mu, a1, a2, deg, hid, n_lb) * theta(rho, mu, a1, a2, deg, hid, n_lb)
    denom = l_val^3 * mu
    return sqrt(num / denom)

# --- Bounds Evaluation ---

def lambda_lb(rho, mu, A, Ceq, deg, hid, n_lb):
    a1, a2 = a_vals(rho, mu, A, Ceq, deg, n_lb)
    l_val = lambda_val(rho, mu)
    h = h_val(rho, mu, a1, a2, deg, hid, n_lb)
    h_term = h + l_val / sigma(mu)
    
    pt1 = C_val(rho, mu, A, Ceq, deg, hid, n_lb) * (h_term^2) * a1 * a2
    pt2 = sqrt(omega(rho, mu, a1, a2, deg, hid, n_lb) * theta(rho, mu, a1, a2, deg, hid, n_lb)) * h_term
    pt3 = log(C_prime(rho, mu, A, Ceq, deg, hid, n_lb) * (h_term^2) * a1 * a2)
    return -pt1 - pt2 - pt3

def lambda_ub(A, Ceq):
    if A == 1 and Ceq == 15:
        return log(2) - (n / 2) * logc
    elif A == 5 and Ceq == 3:
        return 1/2 * log(68 / 25) - (n / 2) * logc

def find_n_bound(A, Ceq, c_lb, n_lb, rholb, rhoub, mu, deg):
    """
    Evaluates the linear forms in logarithms across the specified rho range
    to find the minimal absolute upper bound on n.
    """
    min_n_val = 100000
    min_rho = None
    
    for rho_int in range(int(10 * rholb), int(10 * rhoub) + 1):
        rho = rho_int / 10
        a1_temp, a2_temp = a_vals(rho, mu, A, Ceq, deg, n_lb)
        
        if not verify_lambda(rho, mu, a1_temp, a2_temp, c_lb):
            continue
            
        n_bounds = []
        for hid in range(2):
            lb = lambda_lb(rho, mu, A, Ceq, deg, hid, n_lb)
            ub = lambda_ub(A, Ceq)
            ineq = (ub - lb)(logc = c_lb)
            
            # Solve L2(n) - L1(n) = 0
            root_val = ineq.find_root(10, exp(50 * ln(10)))
            n_bounds.append(floor(root_val).next_prime())
            
        current_max_bound = max(n_bounds)
        if current_max_bound < min_n_val:
            min_n_val = current_max_bound
            min_rho = rho.n()
            
    return min_n_val, min_rho

# =============================================================================
# Execution Pipeline
# =============================================================================

print("=" * 65)
print("Bounds on n for k = 9 (Laurent's Theorem)")
print("=" * 65)

# Parameters extracted from Section 8.2
c_lower_bound = log(121) 
n_lower_bound = 1000
deg_val = 1
mu_values = [1/3, 4/9, 5/9, 2/3, 7/9, 8/9, 1]

for A, Ceq in [(5, 3), (1, 15)]:
    print(f"\nEvaluating (C, D) = ({A}, {Ceq})")
    print("-" * 65)
    print(f"{'mu':<8} | {'Optimal rho':<15} | {'Bound (n < n_0)'}")
    print("-" * 65)
    
    for mu_val in mu_values:
        bound, optimal_rho = find_n_bound(
            A=A, Ceq=Ceq, 
            c_lb=c_lower_bound, n_lb=n_lower_bound, 
            rholb=2, rhoub=20, 
            mu=mu_val, deg=deg_val
        )
        
        # Format the fractions for clean output
        mu_str = str(mu_val)
        rho_str = f"{float(optimal_rho):.1f}" if optimal_rho else "N/A"
        
        print(f"{mu_str:<8} | {rho_str:<15} | {bound}")

Bounds on n for k = 9 (Laurent's Theorem)

Evaluating (C, D) = (5, 3)
-----------------------------------------------------------------
mu       | Optimal rho     | Bound (n < n_0)
-----------------------------------------------------------------
1/3      | 15.1            | 1549
4/9      | 15.4            | 1553
5/9      | 15.1            | 1571
2/3      | 15.7            | 1619
7/9      | 14.6            | 1721
8/9      | 15.6            | 1873
1        | 15.6            | 2111

Evaluating (C, D) = (1, 15)
-----------------------------------------------------------------
mu       | Optimal rho     | Bound (n < n_0)
-----------------------------------------------------------------
1/3      | 14.0            | 1163
4/9      | 15.5            | 1163
5/9      | 14.7            | 1181
2/3      | 14.2            | 1223
7/9      | 14.3            | 1297
8/9      | 13.4            | 1423
1        | 14.1            | 1597


### Local Methods for $k=9$ ($7 < n < n_0$)

As established in the manuscript, when $k=9$ and $p \nmid T_9(x)$, the problem reduces to solving the ternary equation $2C^2c^{2n} + Cc^n + 2 = Dd^n$ for $(C,D) \in \{(5,3), (1,15)\}$. While Baker's method provides an absolute upper bound $n_0$ for the exponent $n$, the remaining search space for $n$ is still large.

To rule out integer solutions for primes $7 < n < n_0$, we rule out the presence of solutions using a local reduction. For each prime $n$, we search for an auxiliary prime $q$ of the form $q = 2mn + 1$. Working modulo $q$, the elements $c^n$ and $d^n$ must either be congruent to $0$ or belong to the set of $m$-th roots of unity in the finite field $\mathbb{F}_q$. 

The script below searches for these auxiliary primes and computes the possible values for $d^n$ given all valid states of $c^n \pmod q$. If no elements in the resulting set map to a valid $n$-th power in $\mathbb{F}_q$, the equation has no solutions modulo $q$, thereby proving it has no integer solutions for that specific exponent $n$.

In [ ]:
# =============================================================================
# Local Methods for k = 9
# =============================================================================

def next_prime_1_mod(lb, n):
    """
    Returns the smallest prime q > lb such that q = 1 (mod 2n).
    This corresponds to auxiliary primes of the form q = 2mn + 1.
    """
    l_val = ceil(lb / (2 * n))
    while True:
        q = 2 * l_val * n + 1
        if q.is_prime():
            return q
        l_val += 1

def get_mu_m(q, n):
    """
    Returns the set of m-th roots of unity in the finite field GF(q),
    which corresponds to the valid n-th powers modulo q.
    """
    m = (q - 1) // n
    # Extract the m-th roots of unity using SageMath's native GF(q) methods
    return set(GF(q)(1).nth_root(m, all=True))

def get_valid_deltas(q, mu_m, C, D):
    """
    Evaluates the ternary equation 2C^2c^{2n} + Cc^n + 2 = Dd^n over GF(q).
    Returns a subset of valid n-th powers (delta) for c^n that yield a 
    valid n-th power for d^n.
    """
    valid_deltas = set()
    
    # Delta (representing c^n) must be an m-th root of unity or 0
    mu_m_with_zero = mu_m.union({GF(q)(0)})
    
    for delta in mu_m_with_zero:
        # Evaluate the generalized polynomial in GF(q)
        d_n_val = (GF(q)(2) * (GF(q)(C) * delta)^2 + GF(q)(C) * delta + GF(q)(2)) / GF(q)(D)
        
        # If the resulting d^n is also a valid n-th power, keep the candidate
        if d_n_val in mu_m_with_zero:
            valid_deltas.add(delta)
            
    return valid_deltas

def local_method_check(n_bound, C, D):
    """
    Rules out integer solutions for all primes 3 <= n < n_bound by 
    finding local contradictions modulo auxiliary primes q.
    """
    print(f"Running local methods for (C, D) = ({C:<2}, {D:<2}) up to n = {n_bound}...")
    
    n = 3
    while n < n_bound:
        q = 1
        
        # Cap the search limit to prevent infinite hangs on resistant cases
        while q < 500 * n:
            q = next_prime_1_mod(q, n)
            mu_m = get_mu_m(q, n)
            valid_deltas = get_valid_deltas(q, mu_m, C, D)
            
            if not valid_deltas:
                # Successfully ruled out modulo q; move to the next prime n
                break
                
        if q > 500 * n:
            print(f"  [WARNING] Trouble proving no solutions for n = {n}. Exceeded prime search limit.")
            
        n = n.next_prime()
        
    print(f"  [SUCCESS] All primes n < {n_bound} ruled out for (C, D) = ({C}, {D}).\n")

# =============================================================================
# Execution Pipeline
# =============================================================================

print("=" * 65)
print("Verifying Local Obstructions for k = 9")
print("=" * 65 + "\n")

# Check bounds established by Laurent's Theorem
local_method_check(1567, C=5, D=3)
local_method_check(1181, C=1, D=15)

Verifying Local Obstructions for k = 9

Running local methods for (C, D) = (5 , 3 ) up to n = 1567...
  [WARNING] Trouble proving no solutions for n = 5. Exceeded prime search limit.
  [SUCCESS] All primes n < 1567 ruled out for (C, D) = (5, 3).

Running local methods for (C, D) = (1 , 15) up to n = 1181...
  [SUCCESS] All primes n < 1181 ruled out for (C, D) = (1, 15).



### Resolving Low Values of $n$ for $k=9$ ($n \leq 7$)

In Section 8.4 of the manuscript, the remaining cases for $k=9$ with small prime exponents ($n \in \{3, 5, 7\}$) are solved by factoring the equation $x^2+x-1=Cc^n$ over the algebraic number field $\mathbb{Q}(\sqrt{5})$. 

For $C \in \{1, 5\}$, this factorization yields a family of binomial Thue equations in the integer variables $V$ and $U$
$$2^n(-1)^r = F_{1-r}V_n - L_{1-r}U_n \quad\text{when }  C=1,$$
$$2^n(-1)^r = 5F_{1-r}U_n - L_{1-r}V_n \quad\text{when } C=5.$$
    
The script below generates these Thue equations for $n \in \{3, 5, 7\}$ and $r \in [-\frac{n-1}{2}, \frac{n+1}{2}]$, solves them unconditionally using PARI/GP, and reverses the change of variables. The verification step calculates $Aa^n$ to isolate the specific solution $x=2$ that yields the near-solution $1^9 + 2^9 = 19 \cdot 3^3$.

In [3]:
# =============================================================================
# Thue Equation Resolutions for k = 9 (n <= 7)
# =============================================================================

# Define the binary recurrence sequences for Fibonacci and Lucas numbers
F = BinaryRecurrenceSequence(1, 1)
L = BinaryRecurrenceSequence(1, 1, 2, 1)

R.<X, Y> = PolynomialRing(QQ)

def S(k, x_val):
    """Returns S_k(x) evaluated at x_val."""
    var('x')
    poly = (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )
    return ZZ(poly(x=x_val))

def solve_thue_unconditional(f, m):
    """Unconditionally solves the Thue equation f = m without GRH."""
    assert f.is_homogeneous()
    parithueinit = gp.thueinit(f.subs({f.variables()[1]: 1}), flag=1)
    return gp.thue(parithueinit, m).sage()

def generate_thue_forms(n, r):
    """Generates the Thue equations for C=1 and C=5 over Q(sqrt(5))."""
    Fr, Lr = F(1 - r), L(1 - r)
    c1, c2 = (X + sqrt(5) * Y), (X - sqrt(5) * Y)
    
    Vn = R(expand((c1^n + c2^n) / 2))
    Un = R(expand((c1^n - c2^n) / (2 * sqrt(5))))
    
    form_c1 = (-1)^r * (Fr * Vn - Lr * Un)
    form_c5 = (-1)^r * (5 * Fr * Un - Lr * Vn)
    
    return {1: form_c1, 5: form_c5}

def verify_solution(n, r, C, V, U):
    """
    Reverses the change of variables over Q(sqrt(5)) to evaluate A*a^n.
    Returns the expanded value.
    """
    eps = (1 + sqrt(5)) / 2
    eps_inv = (-1 + sqrt(5)) / 2
    theta = 1 if C == 1 else sqrt(5)
    
    if r - 1 >= 0:
        base_term = eps^(r - 1)
    else:
        base_term = eps_inv^(1 - r)
        
    gamma = (V + U * sqrt(5)) / 2
    Aan = expand(theta * base_term * (gamma^n) - eps_inv)
    
    return Aan

def resolve_k9_low_n():
    """
    Orchestrates the generation and resolution of the Thue equations, 
    and automatically verifies any candidate solutions found.
    """
    print("=" * 75)
    print("Resolving Low n Cases for k = 9 (n <= 7)")
    print("=" * 75 + "\n")
    
    candidate_solutions = []
    
    for n in [3, 5, 7]:
        for r in range(-(n - 1) // 2, (n + 1) // 2 + 1):
            forms = generate_thue_forms(n, r)
            
            for C, form in forms.items():
                sols = solve_thue_unconditional(form, 2^n)
                if sols:
                    for sol in sols:
                        V, U = sol[0], sol[1]
                        candidate_solutions.append((n, r, C, V, U))
                        print(f"Solution found: n={n}, r={str(r).rjust(2)}, C={C} -> [V, U] = [{V}, {U}]")
                        
    print("\n" + "=" * 75)
    print("Verifying Candidate Solutions (Checking Aa^n)")
    print("=" * 75)
    
    for n, r, C, V, U in candidate_solutions:
        Aan = verify_solution(n, r, C, V, U)
        
        # Print the algebraic evaluation
        print(f"\nn={n}, r={str(r).rjust(2)} | C={C} | (V, U)=({str(V).rjust(2)}, {str(U).rjust(2)}) ---> Aa^n = {Aan}")
        
        # Check if this leads to a valid positive integer x
        if Aan in ZZ:
            x_val = Integer(Aan - 1)
            if x_val > 0:
                s_val = S(9, x_val)
                factors = factor(s_val)
                
                # Extract the p * y^n components
                core_part = 1
                y_part = 1
                for p_fac, multiplicity in factors:
                    core_part *= p_fac ** (multiplicity % n)
                    y_part *= p_fac ** (multiplicity // n)
                
                if core_part.is_prime():
                    print(f"  [VALID SOLUTION] x = {x_val}")
                    print(f"  -> S_9({x_val}) = {s_val} = {factors}")
                    print(f"  -> Matches p * y^n: p = {core_part}, y = {y_part}, n = {n}")

# =============================================================================
# Execution Pipeline
# =============================================================================

resolve_k9_low_n()

Resolving Low n Cases for k = 9 (n <= 7)

Solution found: n=3, r=-1, C=1 -> [V, U] = [-2, 0]
Solution found: n=3, r=-1, C=1 -> [V, U] = [1, 1]
Solution found: n=3, r=-1, C=1 -> [V, U] = [7, 1]
Solution found: n=3, r=-1, C=5 -> [V, U] = [1, 1]
Solution found: n=3, r= 0, C=1 -> [V, U] = [-1, -3]
Solution found: n=3, r= 0, C=1 -> [V, U] = [1, 1]
Solution found: n=3, r= 0, C=1 -> [V, U] = [2, 0]
Solution found: n=3, r= 0, C=5 -> [V, U] = [-2, 0]
Solution found: n=3, r= 2, C=1 -> [V, U] = [-1, 3]
Solution found: n=3, r= 2, C=1 -> [V, U] = [1, -1]
Solution found: n=3, r= 2, C=1 -> [V, U] = [2, 0]
Solution found: n=3, r= 2, C=5 -> [V, U] = [2, 0]
Solution found: n=5, r=-2, C=1 -> [V, U] = [1, 1]
Solution found: n=5, r=-1, C=1 -> [V, U] = [-2, 0]
Solution found: n=5, r= 0, C=1 -> [V, U] = [2, 0]
Solution found: n=5, r= 0, C=5 -> [V, U] = [-2, 0]
Solution found: n=5, r= 2, C=1 -> [V, U] = [2, 0]
Solution found: n=5, r= 2, C=5 -> [V, U] = [2, 0]
Solution found: n=5, r= 3, C=1 -> [V, U] = [2, 0]
